In [1]:
import jsonlines
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import numpy as np
import nltk
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import regex as re
from collections import Counter

In [2]:
data_list = []

In [3]:
with jsonlines.open("data/train.jsonl", mode='r') as reader:
    for obj in reader:
        obj.pop('model')
        obj.pop('genre')
        obj.pop('id')
        data_list.append(obj)

In [4]:
len(data_list)

23707

In [5]:
df = pd.DataFrame(data_list)

In [6]:
print(df)

                                                    text  label
0      Duke Ellington, a titan of jazz, revolutionize...      1
1      I reflected on the shifting dynamics of media ...      1
2      In F. Scott Fitzgerald's "The Great Gatsby," t...      1
3      I still chuckle when I think about that time I...      1
4      Yoga, originating in ancient India, encompasse...      1
...                                                  ...    ...
23702  Antioch stands as a pivotal point in the narra...      1
23703  Margaret Cavendish’s The Blazing World stands ...      1
23704  In August of 1793, in a last-ditch effort to w...      0
23705  In interacting with Luciente and others, Conni...      0
23706  Meursault and Clamence, the protagonists from ...      1

[23707 rows x 2 columns]


In [7]:
print(df.text[0])

Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisational prowess of musicians like Nanton. In "Ring Dem Bells," Ellington ingeniously interweaves scat singing as a dialogue with saxophones, creating a dynamic call and response that underscores his vision of music as a fluid conversation between voices and instruments.

Ellington consistently emphasized the voice as an instrument of equal importance to traditional brass and woodwinds, orchestrating his compositions with a keen ear for vocal qualities. This approach is vividly demonstrated in "Mood Indigo," where his orchestration skills shine through non-traditional chord arrangements, transforming the piece into an auditory tapestry of mood and color. It is not merely the notes that define El

In [8]:
tokens_all = [word_tokenize(sentence) for sentence in df.text]

In [9]:
# Removing all tokens consisting only of non-alphanumeric characters, that aren't inbetween words e.g. "battle-field",
# so any tokens such as: "''", "--", ".", "," etc. are removed, but "battle-field" is not removed. 
for sentence in tokens_all:
    while True:
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                sentence.remove(token) # For each token in a sentence, if the token consists only of 1 or more non-alphanumeric characters, remove it from the sentence.

        outs = [] # Creates an empty list. The list is reset to empty every iteration of the while-loop. 
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                outs.append(token) # This for-loop makes an additional check for any non-alphanumeric tokens, and appends then to the empty list 'outs'.

        if outs: # This checks whether the list 'outs' is empty or not. If it's not empty, then 'continue' is activated and the while-loop continues to iterate,
            continue # if the list is empty, then 'break' is activated, and the while-loop terminates, and the outer-most for-loop continues to the next 'sentence' in 'tokens_all'.
        else:
            break

In [10]:
word_list = [[word.lower() for word in sentence] for sentence in tokens_all]

In [11]:
len(word_list)

23707

In [12]:
words = []

In [13]:
for sentence in word_list:
    for word in sentence:
        words.append(word)

In [14]:
len(words)

14949644

In [15]:
vocab = sorted(set(words))

len(vocab)

122750

In [16]:
freq = Counter(words)

In [17]:
stop_words = set(stopwords.words('english'))

In [18]:
len(stop_words)

198

In [19]:
filtered_tokens = [word for word in words if word not in stop_words]

In [20]:
len(filtered_tokens)

8018027

In [21]:
# This keeps the structure of a list with all datapoints in the train set, where each datapoint
# is in the form of a list. So it's a list with 23707 lists/data points with tokens.
# Each token has been converted to small characters with word.lower(). 
filtered_token_list = [[token for token in sentence if token not in stop_words] for sentence in word_list]
# Stop-words have now been removed as well. 

In [22]:
print(filtered_token_list[0])

['duke', 'ellington', 'titan', 'jazz', 'revolutionized', 'genre', 'innovative', 'compositions', 'showcasing', 'remarkable', 'ability', 'integrate', 'voice', 'instrumental', 'music', 'among', 'notable', 'figures', 'contributed', 'artistic', 'symphony', 'ivie', 'anderson', 'whose', 'scat', 'singing', 'mirrored', 'improvisational', 'prowess', 'musicians', 'like', 'nanton', 'ring', 'dem', 'bells', 'ellington', 'ingeniously', 'interweaves', 'scat', 'singing', 'dialogue', 'saxophones', 'creating', 'dynamic', 'call', 'response', 'underscores', 'vision', 'music', 'fluid', 'conversation', 'voices', 'instruments', 'ellington', 'consistently', 'emphasized', 'voice', 'instrument', 'equal', 'importance', 'traditional', 'brass', 'woodwinds', 'orchestrating', 'compositions', 'keen', 'ear', 'vocal', 'qualities', 'approach', 'vividly', 'demonstrated', 'mood', 'indigo', 'orchestration', 'skills', 'shine', 'non-traditional', 'chord', 'arrangements', 'transforming', 'piece', 'auditory', 'tapestry', 'mood'

In [23]:
df['text'] = filtered_token_list

In [24]:
print(df)

                                                    text  label
0      [duke, ellington, titan, jazz, revolutionized,...      1
1      [reflected, shifting, dynamics, media, coverag...      1
2      [f., scott, fitzgerald, 's, great, gatsby, use...      1
3      [still, chuckle, think, time, bombed, midterm,...      1
4      [yoga, originating, ancient, india, encompasse...      1
...                                                  ...    ...
23702  [antioch, stands, pivotal, point, narrative, e...      1
23703  [margaret, cavendish, blazing, world, stands, ...      1
23704  [august, 1793, last-ditch, effort, win, rebel,...      0
23705  [interacting, luciente, others, connie, strugg...      0
23706  [meursault, clamence, protagonists, albert, ca...      1

[23707 rows x 2 columns]


In [25]:
human_train_df = df

human_train_text = []

for i, sentence in human_train_df.iterrows():
    if sentence.label == 0:
        human_train_text.append(sentence.text)

len(human_train_text)

9101

In [26]:
print(human_train_text[0])

['private', 'bank', 'corruption', 'ultimately', 'drove', 'nation', 'economy', 'brink', 'destruction', 'speculator', 'greed', 'corruption', 'merely', 'subdivision', 'behind', 'united', 'states', 'financial', 'dilemma', 'common', 'man', 'guilty', 'although', 'common', 'man', 'course', 'control', 'majority', 'finances', 'speculators', 'large', 'portion', 'settlers', 'abused', 'credibility', 'purchasing', 'land', 'improperly', 'managing', 'financial', 'decisions', 'added', 'force', 'pushing', 'america', 'towards', 'panic', '1819', 'americans', 'spent', 'lavishly', 'presumption', 'could', 'afford', 'profits', 'enough', 'push', 'debt', 'came', 'time', 'tax', 'collection', 'citizens', 'rushed', 'borrow', 'withdraw', 'money', 'debt', 'caught', 'party', 'could', 'step', 'help', 'state', 'banks', 'could', 'nothing', 'help', 'funds', 'insufficient', 'without', 'money', 'common', 'man', 'owed', 'begin', 'federal', 'government', 'bank', 're-chartered', 'two', 'years', 'ago', 'got', 'money', 'specul

In [27]:
human_train_words = []

for sentence in human_train_text:
    for word in sentence:
        human_train_words.append(word)

len(human_train_words)

3184482

In [28]:
human_train_vocab = sorted(set(human_train_words))
len(human_train_vocab)

97765

In [29]:
llm_train_df = df

llm_train_text = []

for i, sentence in llm_train_df.iterrows():
    if sentence.label == 1:
        llm_train_text.append(sentence.text)

len(llm_train_text)

14606

In [30]:
print(llm_train_text[0])

['duke', 'ellington', 'titan', 'jazz', 'revolutionized', 'genre', 'innovative', 'compositions', 'showcasing', 'remarkable', 'ability', 'integrate', 'voice', 'instrumental', 'music', 'among', 'notable', 'figures', 'contributed', 'artistic', 'symphony', 'ivie', 'anderson', 'whose', 'scat', 'singing', 'mirrored', 'improvisational', 'prowess', 'musicians', 'like', 'nanton', 'ring', 'dem', 'bells', 'ellington', 'ingeniously', 'interweaves', 'scat', 'singing', 'dialogue', 'saxophones', 'creating', 'dynamic', 'call', 'response', 'underscores', 'vision', 'music', 'fluid', 'conversation', 'voices', 'instruments', 'ellington', 'consistently', 'emphasized', 'voice', 'instrument', 'equal', 'importance', 'traditional', 'brass', 'woodwinds', 'orchestrating', 'compositions', 'keen', 'ear', 'vocal', 'qualities', 'approach', 'vividly', 'demonstrated', 'mood', 'indigo', 'orchestration', 'skills', 'shine', 'non-traditional', 'chord', 'arrangements', 'transforming', 'piece', 'auditory', 'tapestry', 'mood'

In [31]:
llm_train_words = []

for sentence in llm_train_text:
    for word in sentence:
        llm_train_words.append(word)

len(llm_train_words)

4833545

In [32]:
llm_train_vocab = sorted(set(llm_train_words))
len(llm_train_vocab)

77599

In [33]:
in_human_not_in_llm_vocab = [word for word in human_train_vocab if word not in llm_train_vocab]

In [34]:
len(in_human_not_in_llm_vocab)

44996

In [35]:
in_llm_not_in_human_vocab = [word for word in llm_train_vocab if word not in human_train_vocab]

In [36]:
len(in_llm_not_in_human_vocab)

24830

In [ ]:
### !!! NEXT !!! ###

In [37]:
filtered_token_strings = [" ".join(sentence) for sentence in filtered_token_list]

In [38]:
print(filtered_token_strings[0])

duke ellington titan jazz revolutionized genre innovative compositions showcasing remarkable ability integrate voice instrumental music among notable figures contributed artistic symphony ivie anderson whose scat singing mirrored improvisational prowess musicians like nanton ring dem bells ellington ingeniously interweaves scat singing dialogue saxophones creating dynamic call response underscores vision music fluid conversation voices instruments ellington consistently emphasized voice instrument equal importance traditional brass woodwinds orchestrating compositions keen ear vocal qualities approach vividly demonstrated mood indigo orchestration skills shine non-traditional chord arrangements transforming piece auditory tapestry mood color merely notes define ellington 's genius rather way orchestrates elements drawing unique talents band members like nanton hodges williams create symphony voice resonates authenticity purpose composition dusk exemplifies ellington 's adeptness captur

In [39]:
len(filtered_token_strings[0])

1894

In [40]:
print(df)

                                                    text  label
0      [duke, ellington, titan, jazz, revolutionized,...      1
1      [reflected, shifting, dynamics, media, coverag...      1
2      [f., scott, fitzgerald, 's, great, gatsby, use...      1
3      [still, chuckle, think, time, bombed, midterm,...      1
4      [yoga, originating, ancient, india, encompasse...      1
...                                                  ...    ...
23702  [antioch, stands, pivotal, point, narrative, e...      1
23703  [margaret, cavendish, blazing, world, stands, ...      1
23704  [august, 1793, last-ditch, effort, win, rebel,...      0
23705  [interacting, luciente, others, connie, strugg...      0
23706  [meursault, clamence, protagonists, albert, ca...      1

[23707 rows x 2 columns]


In [41]:
df['text'] = filtered_token_strings

In [42]:
print(df)

                                                    text  label
0      duke ellington titan jazz revolutionized genre...      1
1      reflected shifting dynamics media coverage per...      1
2      f. scott fitzgerald 's great gatsby use color ...      1
3      still chuckle think time bombed midterm exam m...      1
4      yoga originating ancient india encompasses div...      1
...                                                  ...    ...
23702  antioch stands pivotal point narrative early c...      1
23703  margaret cavendish blazing world stands pionee...      1
23704  august 1793 last-ditch effort win rebel leader...      0
23705  interacting luciente others connie struggles u...      0
23706  meursault clamence protagonists albert camus '...      1

[23707 rows x 2 columns]


In [43]:
len(words)

14949644

In [44]:
len(filtered_token_strings)

23707

In [45]:
len(filtered_tokens)

8018027

In [46]:
filtered_vocab = sorted(set(filtered_tokens))

In [47]:
len(filtered_vocab)

122595

In [49]:
tags = pos_tag(filtered_tokens, tagset='universal')

In [50]:
tag_list = [pos_tag(sentence, tagset='universal') for sentence in filtered_token_list]

In [51]:
len(tag_list)

23707

In [52]:
sentence_tags = [[token[-1] for token in sentence] for sentence in tag_list]

In [53]:
print(sentence_tags[0])

['ADJ', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'ADP', 'ADJ', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'PRON', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'ADP', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'VERB', 'ADV', 'NOUN', 'VERB', 'VERB', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'VERB', 'ADV', 'VERB', 'NOUN', 'NOUN', 'ADJ', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'ADJ', 'ADJ', 'ADJ', 'NOUN', 'VERB', 'ADV', 'VERB', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'ADV', 'VERB', 'ADJ', 'NOUN', 'PRT', 'NOUN', 'ADV', 'NOUN', 'VERB', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'VERB', 'NOUN', 'ADP', 'ADJ', 'NOUN', 'VERB', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'ADJ', 'NOUN', 'NOUN', 'VERB', 'VERB', 'PRT', 'ADJ', 'VERB', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'VERB', 'NOUN', 'NOUN', 'NOUN', 'ADJ', 'VERB', 'AD

In [54]:
unique_tags = set([tag[-1] for tag in tags])

In [55]:
len(unique_tags)

12

In [56]:
tags_freq = [dict(Counter(sentence)) for sentence in sentence_tags]

In [57]:
possible_tags = list(unique_tags)

In [58]:
print(possible_tags)

['ADP', 'NUM', 'CONJ', 'PRON', '.', 'VERB', 'NOUN', 'DET', 'PRT', 'X', 'ADJ', 'ADV']


In [59]:
missing_tags = [{tag: 0 for tag in possible_tags if tag not in sentence} for sentence in tags_freq]

In [60]:
len(missing_tags)

23707

In [61]:
print(missing_tags[0])

{'NUM': 0, 'CONJ': 0, '.': 0, 'DET': 0, 'X': 0}


In [62]:
print(tags_freq[0])

{'ADJ': 50, 'NOUN': 112, 'VERB': 47, 'ADP': 6, 'PRON': 1, 'ADV': 7, 'PRT': 3}


In [63]:
all_tags = []

for i in range(len(missing_tags)):
    tags = tags_freq[i]
    missing = missing_tags[i]
    tags.update(missing)
    all_tags.append(tags)

In [64]:
print(all_tags[0])

{'ADJ': 50, 'NOUN': 112, 'VERB': 47, 'ADP': 6, 'PRON': 1, 'ADV': 7, 'PRT': 3, 'NUM': 0, 'CONJ': 0, '.': 0, 'DET': 0, 'X': 0}


In [65]:
# Sorting the dict, 'example_tags', by making a dict comprehension and using sorted() with lambda
# inserted in the 'key'-argument of the sorted() function.
tags_sorted = [{k: v for k, v in sorted(sentence.items(), key=lambda item: item[0])} for sentence in all_tags]

# key=lambda item: item[0] is exactly the same as writing:
# key = def lambda(item):
#           return item[0]

# lambda is a throwaway function, just like _ is a throwaway variable.

In [66]:
print(tags_sorted[0])

{'.': 0, 'ADJ': 50, 'ADP': 6, 'ADV': 7, 'CONJ': 0, 'DET': 0, 'NOUN': 112, 'NUM': 0, 'PRON': 1, 'PRT': 3, 'VERB': 47, 'X': 0}


In [67]:
len(tags_sorted)

23707

In [68]:
tag_values = [list(sentence.values()) for sentence in tags_sorted]

In [69]:
len(tag_values)

23707

In [70]:
print(tag_values[:2])

[[0, 50, 6, 7, 0, 0, 112, 0, 1, 3, 47, 0], [0, 59, 5, 14, 0, 1, 137, 2, 0, 1, 83, 1]]


In [71]:
print(df)

                                                    text  label
0      duke ellington titan jazz revolutionized genre...      1
1      reflected shifting dynamics media coverage per...      1
2      f. scott fitzgerald 's great gatsby use color ...      1
3      still chuckle think time bombed midterm exam m...      1
4      yoga originating ancient india encompasses div...      1
...                                                  ...    ...
23702  antioch stands pivotal point narrative early c...      1
23703  margaret cavendish blazing world stands pionee...      1
23704  august 1793 last-ditch effort win rebel leader...      0
23705  interacting luciente others connie struggles u...      0
23706  meursault clamence protagonists albert camus '...      1

[23707 rows x 2 columns]


In [72]:
df['POS_vector'] = tag_values

In [73]:
print(df)

                                                    text  label  \
0      duke ellington titan jazz revolutionized genre...      1   
1      reflected shifting dynamics media coverage per...      1   
2      f. scott fitzgerald 's great gatsby use color ...      1   
3      still chuckle think time bombed midterm exam m...      1   
4      yoga originating ancient india encompasses div...      1   
...                                                  ...    ...   
23702  antioch stands pivotal point narrative early c...      1   
23703  margaret cavendish blazing world stands pionee...      1   
23704  august 1793 last-ditch effort win rebel leader...      0   
23705  interacting luciente others connie struggles u...      0   
23706  meursault clamence protagonists albert camus '...      1   

                                       POS_vector  
0        [0, 50, 6, 7, 0, 0, 112, 0, 1, 3, 47, 0]  
1       [0, 59, 5, 14, 0, 1, 137, 2, 0, 1, 83, 1]  
2       [0, 73, 7, 13, 0, 0, 141, 1, 0,

In [74]:
pos_vectors = [[number/np.sum(values) for number in values] for values in tag_values]

In [75]:
np.set_printoptions(legacy='1.25')

In [76]:
df['POS_vector'] = pos_vectors

In [77]:
print(df)

                                                    text  label  \
0      duke ellington titan jazz revolutionized genre...      1   
1      reflected shifting dynamics media coverage per...      1   
2      f. scott fitzgerald 's great gatsby use color ...      1   
3      still chuckle think time bombed midterm exam m...      1   
4      yoga originating ancient india encompasses div...      1   
...                                                  ...    ...   
23702  antioch stands pivotal point narrative early c...      1   
23703  margaret cavendish blazing world stands pionee...      1   
23704  august 1793 last-ditch effort win rebel leader...      0   
23705  interacting luciente others connie struggles u...      0   
23706  meursault clamence protagonists albert camus '...      1   

                                              POS_vector  
0      [0.0, 0.22123893805309736, 0.02654867256637168...  
1      [0.0, 0.19471947194719472, 0.0165016501650165,...  
2      [0.0, 0.245

!!!-----------------DEV DATASET-----------------!!!

In [78]:
dev_data = []

with jsonlines.open("data/dev.jsonl", mode='r') as doc:
    for obj in doc:
        obj.pop('model')
        obj.pop('genre')
        obj.pop('id')
        dev_data.append(obj)


In [79]:
df_dev = pd.DataFrame(dev_data)
len(df_dev)

1795

In [80]:
print(df_dev)

                                                   text  label
0     Harry felt a sense of excitement as he walked ...      1
1     Squire Stretford was one of the good old sort ...      0
2     I look around and see the diversity of familie...      1
3     The young man returned home in a daze, his min...      1
4     The young officer's heart pounded in his chest...      1
...                                                 ...    ...
1790  Dora had the same yellow curly hair, in tight,...      0
1791  The air was thick with the hum of voices, the ...      1
1792  "My poor Dorothy! I do indeed feel for you."\n...      0
1793  Washington Post Staff Escapes Kabul Amidst Det...      1
1794  She at last acknowledged to Mr. Walker that sh...      0

[1795 rows x 2 columns]


In [81]:
dev_tokens = [word_tokenize(sentence) for sentence in df_dev.text]

In [82]:
# Removing all tokens consisting only of non-alphanumeric characters, that aren't inbetween words e.g. "battle-field",
# so any tokens such as: "''", "--", ".", "," etc. are removed, but "battle-field" is not removed. 
for sentence in dev_tokens:
    while True:
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                sentence.remove(token) # For each token in a sentence, if the token consists only of 1 or more non-alphanumeric characters, remove it from the sentence.

        outs = [] # Creates an empty list. The list is reset to empty every iteration of the while-loop. 
        for token in sentence:
            if re.search(r"\B[^a-zA-Z0-9]+\B", token) != None:
                outs.append(token) # This for-loop makes an additional check for any non-alphanumeric tokens, and appends then to the empty list 'outs'.

        if outs: # This checks whether the list 'outs' is empty or not. If it's not empty, then 'continue' is activated and the while-loop continues to iterate,
            continue # if the list is empty, then 'break' is activated, and the while-loop terminates, and the outer-most for-loop continues to the next 'sentence' in 'tokens_all'.
        else:
            break

In [83]:
word_list_dev = [[word.lower() for word in sentence] for sentence in dev_tokens]

In [84]:
words_dev = []

In [85]:
for sentence in word_list_dev:
    for word in sentence:
        words_dev.append(word)

In [86]:
len(words_dev)

1109523

In [87]:
vocab_dev = sorted(set(words_dev))

In [88]:
len(vocab_dev)

41666

In [89]:
filtered_tokens_dev = [word for word in words_dev if word not in stop_words]

In [90]:
len(filtered_tokens_dev)

604723

In [91]:
# This keeps the structure of a list with all datapoints in the train set, where each datapoint
# is in the form of a list. So it's a list with 23707 lists/data points with tokens.
# Each token has been converted to small characters with word.lower(). 
filtered_token_list_dev = [[token for token in sentence if token not in stop_words] for sentence in word_list_dev]
# Stop-words have now been removed as well. 

In [92]:
filtered_token_strings_dev = [" ".join(sentence) for sentence in filtered_token_list_dev]

In [93]:
df_dev['text'] = filtered_token_strings_dev

In [94]:
print(df_dev)

                                                   text  label
0     harry felt sense excitement walked towards new...      1
1     squire stretford one good old sort country gen...      0
2     look around see diversity families life single...      1
3     young man returned home daze mind swirling con...      1
4     young officer 's heart pounded chest followed ...      1
...                                                 ...    ...
1790  dora yellow curly hair tight frizzly rings hea...      0
1791  air thick hum voices rustle silken gowns faint...      1
1792  poor dorothy indeed feel knew saw headless hor...      0
1793  washington post staff escapes kabul amidst det...      1
1794  last acknowledged mr. walker could account twe...      0

[1795 rows x 2 columns]


In [95]:
tags_dev = pos_tag(filtered_tokens_dev, tagset='universal')

In [96]:
len(tags_dev)

604723

In [97]:
tag_list_dev = [pos_tag(sentence, tagset='universal') for sentence in filtered_token_list_dev]

In [98]:
len(tag_list_dev)

1795

In [99]:
sentence_tags_dev = [[token[-1] for token in sentence] for sentence in tag_list_dev]

In [100]:
unique_tags_dev = set([tag[-1] for tag in tags_dev])

In [101]:
len(unique_tags_dev)

12

In [102]:
tags_freq_dev = [dict(Counter(sentence)) for sentence in sentence_tags_dev]

In [103]:
missing_tags_dev = [{tag: 0 for tag in possible_tags if tag not in sentence} for sentence in tags_freq_dev]

In [104]:
len(missing_tags_dev)

1795

In [105]:
all_tags_dev = []

for i in range(len(missing_tags_dev)):
    tags_dev = tags_freq_dev[i]
    missing_dev = missing_tags_dev[i]
    tags_dev.update(missing_dev)
    all_tags_dev.append(tags_dev)

In [106]:
# Sorting the dict, 'example_tags', by making a dict comprehension and using sorted() with lambda
# inserted in the 'key'-argument of the sorted() function.
tags_sorted_dev = [{k: v for k, v in sorted(sentence.items(), key=lambda item: item[0])} for sentence in all_tags_dev]

# key=lambda item: item[0] is exactly the same as writing:
# key = def lambda(item):
#           return item[0]

# lambda is a throwaway function, just like _ is a throwaway variable.

In [107]:
len(tags_sorted_dev)

1795

In [108]:
tag_values_dev = [list(sentence.values()) for sentence in tags_sorted_dev]

In [109]:
len(tag_values_dev)

1795

In [110]:
print(tag_values_dev[:2])

[[0, 49, 8, 15, 0, 1, 134, 4, 0, 10, 81, 0], [0, 68, 3, 27, 0, 0, 144, 5, 0, 1, 95, 0]]


In [111]:
pos_vectors_dev = [[number/np.sum(values) for number in values] for values in tag_values_dev]

In [112]:
df_dev['POS_vector'] = pos_vectors_dev

In [113]:
print(pos_vectors_dev[0])

[0.0, 0.16225165562913907, 0.026490066225165563, 0.04966887417218543, 0.0, 0.0033112582781456954, 0.44370860927152317, 0.013245033112582781, 0.0, 0.033112582781456956, 0.2682119205298013, 0.0]


In [114]:
print(df_dev)

                                                   text  label  \
0     harry felt sense excitement walked towards new...      1   
1     squire stretford one good old sort country gen...      0   
2     look around see diversity families life single...      1   
3     young man returned home daze mind swirling con...      1   
4     young officer 's heart pounded chest followed ...      1   
...                                                 ...    ...   
1790  dora yellow curly hair tight frizzly rings hea...      0   
1791  air thick hum voices rustle silken gowns faint...      1   
1792  poor dorothy indeed feel knew saw headless hor...      0   
1793  washington post staff escapes kabul amidst det...      1   
1794  last acknowledged mr. walker could account twe...      0   

                                             POS_vector  
0     [0.0, 0.16225165562913907, 0.02649006622516556...  
1     [0.0, 0.19825072886297376, 0.00874635568513119...  
2     [0.0, 0.22268907563025211, 

In [115]:
# Examine your Vocabulary (e.g., sort by frequency, alphabet, etc.).
#    a. What kinds of words do you see?
#    b. Does your text normalization seem to work well? Why or why not?
#    c. Are there marked differences between human_Vocab and
#       machine_Vocab?
#    d. What % of tokens were seen in train but not dev?

In [116]:
filtered_vocab_dev = sorted(set(filtered_tokens_dev))

In [117]:
len(filtered_vocab_dev)

41514

In [118]:
len(filtered_vocab)

122595

In [119]:
missing_tokens_dev = [token for token in filtered_vocab_dev if token not in filtered_vocab]

In [120]:
len(missing_tokens_dev)

3743

In [121]:
# Percentage of unknown tokens in the dev set.
3743 / 41514 * 100

9.016235486823723

Training Logistic Regression on TF-IDF features and POS tag-vectors

In [122]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=5, ngram_range=(1, 2))

In [124]:
train_tfidf_features = tfidf.fit_transform(df.text).toarray()

In [138]:
print(train_tfidf_features.shape)

(23707, 229883)


In [152]:
labels_train = df.label

In [149]:
pos_features = df.POS_vector

In [151]:
features_train = pos_features

In [128]:
dev_tfidf_features = tfidf.fit_transform(df_dev.text).toarray()

In [129]:
print(dev_tfidf_features.shape)

(1795, 17712)


In [131]:
labels_dev = df_dev.label

In [132]:
features_dev = [dev_tfidf_features, df_dev.POS_vector]

In [153]:
logreg = LogisticRegression(max_iter=1000, random_state=42)

In [154]:
logreg.fit(features_train, labels_train)

ValueError: setting an array element with a sequence.